<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_2_1_1b_LogReg_Model_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Evaluation for Logistic Regression

Author: Brad Sheese

---

## Introduction

In this notebook, we dive deep into evaluating classification models.

In [ ]:
# Download the cleaning module
import urllib.request
module_url = "https://raw.githubusercontent.com/bsheese/cs377/main/18_classification/classification_cleaning.py"
urllib.request.urlretrieve(module_url, "classification_cleaning.py")
from classification_cleaning import load_and_clean_titanic

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

X_train, X_test, y_train, y_test, features = load_and_clean_titanic(test_size=0.3)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 1. Confusion Matrix Deep Dive

The confusion matrix breaks down predictions into four categories.

In [ ]:
# Create and visualize confusion matrix
cm = confusion_matrix(y_test, pipeline.predict(X_test))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Died', 'Survived'],
            yticklabels=['Died', 'Survived'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()
print(classification_report(y_test, pipeline.predict(X_test), target_names=['Died', 'Survived']))

## 2. ROC Curve and AUC

The ROC curve visualizes the trade-off between True Positive Rate and False Positive Rate.

In [ ]:
# Calculate ROC curve
y_proba = pipeline.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid()
plt.show()
print(f"AUC: {roc_auc:.3f}")

## 3. Precision-Recall Curve

Useful for imbalanced datasets.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, _ = precision_recall_curve(y_test, y_proba)
avg_precision = average_precision_score(y_test, y_proba)

plt.figure(figsize=(10, 8))
plt.plot(recall, precision, lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title(f'Precision-Recall Curve (AP = {avg_precision:.3f})')
plt.grid()
plt.show()

## 4. Threshold Tuning

Adjust the decision threshold based on problem context.

In [ ]:
from sklearn.metrics import precision_score, recall_score, accuracy_score

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
print("Threshold Analysis:")
for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    print(f"  t={t}: Acc={accuracy_score(y_test, y_pred_t):.3f}, Prec={precision_score(y_test, y_pred_t):.3f}, Rec={recall_score(y_test, y_pred_t):.3f}")

## 5. Cross-Validation

Use k-fold CV for more reliable estimates.

In [ ]:
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
print(f"CV Accuracy: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")
print(f"Individual folds: {cv_scores}")